## Real vs Fake Faces

We are going to `Fake-Vs-Real-Faces`. The Dataset contains two types of image which are: `Fake faces` image collected from the website [thispersondoesnotexist.com](thispersondoesnotexist.com). , `Real faces` images are re collected through the API of the website [Unsplash](https://unsplash.com/) and then the faces are cropped out of using OpenCV library.

Here some additional information about the dataset:
- Total number of images: 1288
- Number of "Fake" faces: 700
- Number of "Real" faces: 589

### 1. Import Libraries 

In [76]:
import torch 
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, random_split
from torchvision.datasets import ImageFolder # Stream data from images stored in folders
import torch.optim as optim
import torch.nn.functional as F

from sklearn.model_selection import StratifiedKFold, train_test_split
import os # allow s to access files
import numpy as np
from PIL import Image # Allows us to Load Images
from collections import Counter # Utility function to give us the counts of unique items in an iterable
import matplotlib.pyplot as plt

### 2. Load and Process the data

First Let's define the transforms pipeline for `data augmentatin` and `preprocessing` steps

In [88]:
torch.manual_seed(42)

In [95]:
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

# for data preprocessing and data augmentation for trianing data
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),

])

Let's load our images using `ImageFolder`. This class inherits from `DatasetFolder`  class. So, we can override the classes to create custom datasets as well.

In [96]:
path_to_data = './data/'
dataset = ImageFolder(path_to_data, transform=train_transforms)

After loading the dataset we need to split it, but first let's define the `split` ratios

In [97]:
# # splite ratios
train_ratio = 0.8
test_ratio = 0.2

# # size of the data
dataset_size = len(dataset)

train_size = int(train_ratio * dataset_size)
test_size = dataset_size - train_size
# test_size = dataset_size - train_size - val_size

check if the data correctly splitted 

In [98]:
dataset_size == test_size + train_size 

True

Now let's splite the data

In [99]:
train_data, test_data = random_split(dataset, [train_size, test_size])

### 3. Build Helper functions

Before starting building the model we will build a helper function to save our models

In [100]:
def save_model(epochs, model, optimizer, loss_fun, model_name):

    """
    Function to save the trained model to disk
    """
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss_fun,
    }, f'outputs/{model_name}.pth')

The next function is for saving the loss and accuracy plots after the training completes

In [101]:
def save_plots(train_acc, valid_acc, test_acc, train_loss, valid_loss, test_loss):

    """
    Function to save the loss and accurcy plots to disk
    """

    # accuracy plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_acc, color='green', linestyle='-', label='train accuracy')
    plt.plot(valid_acc, color='red', linestyle='-', label='validation accuracy')
    plt.plot(test_acc, color='cyan', linestyle='-', label='test accuracy')
    plt.xlabel('EPOCHs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig('outputs/accuracy.png')

    # loss plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_loss, color='orange', linestyle='-', label='train loss')
    plt.plot(valid_loss, color='blue', linestyle='-', label='valid loss')
    plt.plot(test_loss, color='magenta', linestyle='-', label='test loss')
    plt.xlabel('EPOCHs')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('outputs/loss.png')

### 3. Prepparing the Data Loaders and Stratified KFold

Now we will build our `data loader` and `stratified KFold` spliter for that we need for training.

In [102]:
BATCH_SIZE = 16
N_SPLITs = 5
SEED = 42

In [ ]:
def train(model, train_data, optimizer, loss_fn, Epochs):

    

In [106]:
def SKFOLD(model, train_data, optimizer, loss_fn, Epochs = 10):

    """
    Train the given model
    """
    
    skf = StratifiedKFold(n_splits=N_SPLITs, shuffle=True, random_state=SEED)
    stratified_scores = []

    # Lists to collect metrics
    train_acc_list, valid_acc_list, test_acc_list = [], [], []
    train_loss_list, valid_loss_list, test_loss_list = [], [], []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(train_data)):


        # folds results 
        all_fold_results = []
        
        # Create the Data Loaders
        train_subset = Subset(train_data, train_idx)
        val_subset = Subset(train_data, valid__idx)


        # Apply transforms dynamically
        train_subset.dataset.transform = train_transforms
        val_subset.dataset.transform = valid_transforms
        
        train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
        valid_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

        ### Training ###
        model.train() 
        print("Training")
        for epoch in range(Epochs):

            train_loss, correct, total = 0.0, 0, 0
            
            for images, labels in train_loader:
                forward_out = model(images)
                labels = labels.unsqueeze(1).float()
                loss = loss_fn(forward_out, labels)
                loss.backward() # backpagation

                train_loss += loss.items()
                
                optimizer.step() # update the weight
                optimizer.zero_grad() # set the gradient to zero
                
                
                correct += (forward_out == labels).sum().item()
                total += labels.size(0)


            # calculate the accuracy and loss for the current epoch
            train_accuracy = correct / total
            train_acc_list.append(train_accuracy)
            train_loss_list.append(train_loss / len(train_loader))

            ### Validation ###
            model.eval()
            print("Evaluting")
            val_loss, correct, total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in train_loader:
                    forward_out = model(images)
                    loss = loss_fn(forward_out, labels)
                    
                    val_loss += loss.item()
                    correct += (forward_out == labels).sum().item()
                    total += labels.size(0)

                    
            val_accuracy = correct / total
            valid_acc_list.append(val_accuracy)
            valid_loss_list.append(val_loss / len(val_loader))
     

        
            fold_train_loss.append(train_loss)
            fold_val_loss.append(val_loss)
            fold_train_acc.append(train_acc)
            fold_val_acc.append(val_acc)
            
        # Store metrics for this fold
        all_fold_results.append({
            'fold': fold + 1,
            'train_loss': fold_train_loss,
            'val_loss': fold_val_loss,
            'train_acc': fold_train_acc,
            'val_acc': fold_val_acc
        })


    avg_val_acc = sum(r['val_acc'][-1] for r in all_fold_results) / N_SPLITS
    avg_val_loss = sum(r['val_loss'][-1] for r in all_fold_results) / N_SPLITS
    print(f"\n🏁 Average Validation Accuracy across folds: {avg_val_acc:.4f}")
    print(f"🏁 Average Validation Loss across folds: {avg_val_loss:.4f}")

    return all_fold_results
        
              
    

### 4. Build the Models

In [ ]:
import torchvision.models as models

# Load the AlexNet Model with pre-trian ImageNet weights
alexnet = model.alexnet(pretrained=True)

#
alexnet.eval()